In [ ]:
from ollama import chat

# Import der Sätze

In [ ]:
sentences = ["Victoria splendidissima! Dux gloriam aeternam meruit!","Bellum crudele et longum populum afflixerat."]

In [ ]:
import ollama
import pandas as pd

def classify_with_rules(latin_text: str) -> str:
    # Modell-Prediction via ollama library statt subprocess
    response = ollama.chat(
        model='augustulus-latin',
        messages=[{'role': 'user', 'content': latin_text}]
    )
    prediction = response.message.content.strip()
    
    # Linguistic Rules (unverändert)
    text_lower = latin_text.lower()
    
    extreme_neg = ['crudel', 'saev', 'trucidat', 'perdi', 'desperatio']
    extreme_pos = ['splendidissim', 'magnificus', 'beatitudo', 'triumphus magnificus']
    
    has_extreme_neg = any(m in text_lower for m in extreme_neg)
    has_extreme_pos = any(m in text_lower for m in extreme_pos)
    exclamations   = latin_text.count('!')
    
    if 'MODERATELY NEGATIVE' in prediction and has_extreme_neg:
        return 'VERY NEGATIVE'
    if 'MODERATELY POSITIVE' in prediction and has_extreme_pos and exclamations >= 2:
        return 'EXTREMELY POSITIVE'
    
    return prediction

# For-Loop mit Post-Processing
results = []
for sentence in sentences:
    results.append({
        'input':      sentence,
        'output':     classify_with_rules(sentence),
    })

df = pd.DataFrame(results)

In [ ]:
# CSV speichern
df.to_csv('data/sentiment.csv', index=False, encoding='utf-8')